In [ ]:
from __future__ import annotations

import math
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import Tensor
from torch.distributions import Normal
import os
if Path.cwd().name == "notebooks":
    os.chdir("..")

from sazz.gpu_friendly.scripts.uci_bnn_grid import (
    load_raw_datasets, make_split, build_target, BNNConfig, BASE_SEED, DTYPE, DEVICE,
)

## COVERAGE

In [ ]:
VARIANT = "shallow"
RESULTS_DIR = Path("results/paper/") / VARIANT #/ "minibatch_32"

from sazz.gpu_friendly.scripts.uci_bnn_grid import UCI_DATASETS

DATASETS = ("boston",)          # extend as needed
print("Datasets found:", DATASETS)

SAMPLER_LABELS = {
    "grid_zigzag":           "ZigZag",
    "grid_sticky_zigzag":    "Sticky ZigZag",
    "grid_boomerang":        "Boomerang",
    "grid_sticky_boomerang": "Sticky Boomerang",
    "nuts":                  "NUTS",
    "nuts_horseshoe":        "NUTS-HS",
    "lbbnn":                 "LBBNN",
    "tf_boomerang":          "TF Boomerang (Goan)",
}

REFERENCE_SAMPLER = "nuts"
CRPS_MIX_SUBSAMPLE = 256

In [ ]:
print("Loading raw datasets for test-set reconstruction...")
raw = load_raw_datasets(tuple(DATASETS))
print("Done.", list(raw.keys()))

In [ ]:
from sazz.gpu_friendly.scripts.uci_bnn_grid import (
    load_raw_datasets, make_split, BNNConfig, BASE_SEED, DTYPE, DEVICE,
)
from sazz.gpu_friendly.models.neural_networks import FFN
from sazz.gpu_friendly.models.model import BayesianModule
from sazz.gpu_friendly.models.priors import build_fan_in_prior_precision

raw = load_raw_datasets(DATASETS)


@torch.no_grad()
def predict_all(bm, weight_samples: Tensor, X_new: Tensor) -> Tensor:
    return torch.stack([
        torch.func.functional_call(bm.module, bm.param_dict_fn(beta), (X_new,)).squeeze(-1)
        for beta in weight_samples
    ])  # [S, N]


def build_bm(data: dict, cfg: BNNConfig) -> BayesianModule:
    """Posterior target only, identical to build_target's bm, without the MAP / reference search."""
    module = FFN(cfg.layer_sizes, cfg.activation)
    prec = build_fan_in_prior_precision(
        module, cfg.prior_std_weight, cfg.prior_std_bias, cfg.fan_in_scaling,
        dtype=DTYPE, device=DEVICE,
    )
    return BayesianModule.build(
        module, likelihood="gaussian", X=data["X_train"], y=data["y_train"],
        prior_precision=prec, prior_sigma_scale=cfg.prior_sigma_scale,
        dtype=DTYPE, device=DEVICE,
    )


# --- index: (dataset, split_id) -> run files ---------------------------------
SPLITS: dict[tuple[str, int], list[Path]] = {}
for dataset in DATASETS:
    ds_dir = RESULTS_DIR / dataset
    if not ds_dir.exists():
        print(f"  [{dataset}] no results directory found, skipping.")
        continue
    if dataset not in raw:
        print(f"  [{dataset}] not available in raw data, skipping.")
        continue
    for split_dir in sorted(ds_dir.glob("split_*")):
        files = [p for p in sorted(split_dir.glob("*.pt")) if not p.name.endswith("_inclusion.pt")]
        if files:
            SPLITS[(dataset, int(split_dir.name.split("_")[-1]))] = files

print(f"Indexed {len(SPLITS)} split(s):")
for (ds, sid), files in SPLITS.items():
    print(f"  {ds} / split {sid:02d}: {', '.join(p.stem for p in files)}")


def load_split(dataset: str, split_id: int) -> dict:
    """Load every run of one (dataset, split) into the same `runs` payload as before."""
    X_all, y_all = raw[dataset]
    data = make_split(X_all, y_all, seed=BASE_SEED + split_id, dtype=DTYPE, device=DEVICE)
    X_test = data["X_test"]

    bm, runs = None, {}
    for pt in SPLITS[(dataset, split_id)]:
        stem = pt.stem
        run  = torch.load(pt, map_location="cpu", weights_only=False)
        samples = run["samples"].to(dtype=DTYPE)   # [S, D], last column is log_sigma
        if samples.shape[0] == 0:
            print(f"  [{dataset} / split {split_id} / {stem}] SKIP: 0 samples "
                  f"(n_events={run.get('n_events')}, grad_evals={run.get('gradient_evals')})")
            continue

        if bm is None:
            cfg = BNNConfig(
                layer_sizes=run["layer_sizes"],
                activation=run["activation"],
                prior_sigma_scale=run["prior_sigma_scale"],
            )
            bm = build_bm(data, cfg)

        weight_samples = samples[:, :-1]
        preds          = predict_all(bm, weight_samples, X_test)   # [S, N]
        epist_std      = preds.std(0)
        noise_samples  = samples[:, -1].exp()                      # [S]
        noise_std_eff  = float(noise_samples.mean())

        runs[stem] = dict(
            run            = run,
            samples        = samples,
            weight_samples = weight_samples,
            preds          = preds,
            mean_pred      = preds.mean(0),
            epist_std      = epist_std,
            total_std      = (epist_std ** 2 + noise_std_eff ** 2).sqrt(),
            noise_std_eff  = noise_std_eff,
            noise_samples  = noise_samples,
            label          = SAMPLER_LABELS.get(stem, stem),
        )
    return dict(dataset=dataset, split_id=split_id, data=data, bm=bm, runs=runs,
                X_test=X_test, y_test=data["y_test"], y_std=data["y_std"])


def use_split(dataset: str, split_id: int) -> dict:
    """Load one split and expose it under the globals the analysis cells already use."""
    global DATASET, SPLIT_ID, data, bm, runs, X_test, y_test, y_std, sampler_order
    ctx = load_split(dataset, split_id)
    DATASET, SPLIT_ID = dataset, split_id
    data, bm, runs = ctx["data"], ctx["bm"], ctx["runs"]
    X_test, y_test, y_std = ctx["X_test"], ctx["y_test"], ctx["y_std"]
    sampler_order = list(runs.keys())
    print(f"Active split: {dataset} / split {split_id:02d}  ({', '.join(sampler_order)})")
    return ctx


In [ ]:
full = use_split("boston", 0)

runs = full['runs']

In [ ]:
# MAP prediction from x_ref (Adam estimate) -- one evaluation per run to verify x_ref consistency
print("MAP predictions (x_ref):")
for stem, s in runs.items():
    x_ref = s["run"]["x_ref"]
    if x_ref is None:
        print(f"  {s['label']}: no x_ref stored")
        continue
    x_ref = x_ref.to(dtype=DTYPE)
    weights_ref = x_ref[:-1]
    with torch.no_grad():
        pred = torch.func.functional_call(bm.module, bm.param_dict_fn(weights_ref), (X_test,)).squeeze(-1)
    noise = float(x_ref[-1].exp())
    rmse  = float(((pred - y_test) ** 2).mean().sqrt()) * y_std
    print(f"  {s['label']}: RMSE={rmse:.3f}  noise_std={noise:.3f}")

In [ ]:
zigzag_run = runs['grid_zigzag']
boomerang_run = runs['grid_boomerang']
nuts_run = runs['nuts']

s_zigzag_run = runs['grid_sticky_zigzag']
s_boomerang_run = runs['grid_sticky_boomerang']
lbbnn_run = runs['lbbnn']

In [ ]:
model = bm


zz_samples = zigzag_run["samples"].to(dtype=DTYPE)   # [S, D], last column is log_sigma
zz_weight_samples = zz_samples[:, :-1]
boom_samples = boomerang_run["samples"].to(dtype=DTYPE)   # [S, D], last column is log_sigma
boom_weight_samples = boom_samples[:, :-1]
nuts_samples = nuts_run["samples"].to(dtype=DTYPE)   # [S, D], last column is log_sigma
nuts_weight_samples = nuts_samples[:, :-1]


s_zz_samples = s_zigzag_run["samples"].to(dtype=DTYPE)   # [S, D], last column is log_sigma
s_zz_weight_samples = s_zz_samples[:, :-1]
s_boom_samples = s_boomerang_run["samples"].to(dtype=DTYPE)   # [S, D], last column is log_sigma
s_boom_weight_samples = s_boom_samples[:, :-1]
lbbnn_samples = lbbnn_run["samples"].to(dtype=DTYPE)   # [S, D], last column is log_sigma
lbbnn_weight_samples = lbbnn_samples[:, :-1]

zz_preds     = predict_all(model, zz_weight_samples, X_test)   # [S, N]
boom_preds     = predict_all(model, boom_weight_samples, X_test)   # [S, N]
nuts_preds     = predict_all(model, nuts_weight_samples, X_test)   # [S, N]

s_zz_preds     = predict_all(model, s_zz_weight_samples, X_test)   # [S, N]
s_boom_preds     = predict_all(model, s_boom_weight_samples, X_test)   # [S, N]
lbbnn_preds     = predict_all(model, lbbnn_weight_samples, X_test)   # [S, N]

zz_sigma = zigzag_run['noise_samples']
boom_sigma = boomerang_run['noise_samples']
nuts_sigma = nuts_run['noise_samples']

s_zz_sigma = s_zigzag_run['noise_samples']
s_boom_sigma = s_boomerang_run['noise_samples']
lbbnn_sigma = lbbnn_run['noise_samples']

In [ ]:

# ---------------------------------------------------------------------------
# COVERAGE via the probability integral transform (PIT)
# ---------------------------------------------------------------------------
# Model:  y | x, θ, σ  ~  N(f_θ(x), σ²).
# Posterior draws (θ_s, σ_s), s = 1..S, give the posterior predictive at x_i
# as an equally weighted Gaussian mixture
#
#     p(y | x_i, D)  ≈  (1/S) Σ_s N(y ; f_s(x_i), σ_s²),          f_s := f_{θ_s},
#
# with CDF
#
#     F_i(y)  =  (1/S) Σ_s Φ( (y − f_s(x_i)) / σ_s ).
#
# PIT value of the observed test response:  u_i = F_i(y_i).
# If the predictive is calibrated, u_i ~ Uniform(0, 1).
#
# Central (1 − α) predictive interval:  I_i(α) = [F_i⁻¹(α/2), F_i⁻¹(1 − α/2)].
# F_i is continuous and strictly increasing, so
#
#     y_i ∈ I_i(α)   ⇔   α/2 ≤ u_i ≤ 1 − α/2   ⇔   |u_i − 1/2| ≤ p/2,    p := 1 − α.
#
# Empirical coverage at nominal level p is therefore
#
#     C(p)  =  (1/N) Σ_i 1{ |u_i − 1/2| ≤ p/2 },
#
# available for every p from the same u_i. The noise term enters exactly
# through Φ, so there is no simulation of ε. Calibration means C(p) = p for all p.
#
# Sharpness (mean interval width):
#
#     W(p)  =  (1/N) Σ_i [ F_i⁻¹((1+p)/2) − F_i⁻¹((1−p)/2) ],
#
# with F_i⁻¹ found by bisection, which is valid because F_i is monotone. The
# bracket [min_s(f_s − 10σ_s), max_s(f_s + 10σ_s)] contains every quantile of interest.
# Widths are reported on the original response scale, W · s_y, where y was
# standardised as (y − m_y) / s_y.
#
# Reference band: under calibration N·C(p) ~ Bin(N, p), so
#     se(C(p)) = sqrt(p(1 − p) / N).
#
# Note: coverage is a property of (model, posterior approximation), not of the
# sampler alone. An exact posterior of a misspecified model need not reach C(p) = p.
# ---------------------------------------------------------------------------

import matplotlib.pyplot as plt

STD_NORMAL = Normal(0.0, 1.0)

PRED = {                       # stem -> (f_s(x_i) [S, N], σ_s [S])
    "grid_zigzag":           (zz_preds,     zz_sigma),
    "grid_boomerang":        (boom_preds,   boom_sigma),
    "grid_sticky_zigzag":    (s_zz_preds,   s_zz_sigma),
    "grid_sticky_boomerang": (s_boom_preds, s_boom_sigma),
    "nuts":                  (nuts_preds,   nuts_sigma),
    "lbbnn":                 (lbbnn_preds,  lbbnn_sigma),
}

# Colour palette -- one colour per base sampler family
COLORS = {
    "grid_zigzag":           "#15A6DB",
    "grid_sticky_zigzag":    "#15A6DB",
    "grid_boomerang":        "#FA0000",
    "grid_sticky_boomerang": "#FA0000",
    "nuts":                  "#0CCA38",
    "nuts_horseshoe":        "#0CCA38",
    "lbbnn":                "#CE0577",
}

# Linestyle: solid for sticky/HS, dashed for vanilla
LINESTYLES = {
    "grid_zigzag":           "--",
    "grid_sticky_zigzag":    "-",
    "grid_boomerang":        "--",
    "grid_sticky_boomerang": "-",
    "nuts":                  "--",
    "nuts_horseshoe":        "-",
}


def pit_values(f: Tensor, sigma: Tensor, y: Tensor) -> Tensor:
    """u_i = (1/S) Σ_s Φ((y_i − f_s(x_i)) / σ_s).   f: [S, N], σ: [S], y: [N] -> [N]."""
    return STD_NORMAL.cdf((y[None, :] - f) / sigma[:, None]).mean(0)


def predictive_quantiles(f: Tensor, sigma: Tensor, probs: Tensor, n_iter: int = 60) -> Tensor:
    """Solve F_i(q) = π by bisection for every π in probs and every i.  -> [len(probs), N]."""
    L, N = probs.shape[0], f.shape[1]
    lo = (f - 10 * sigma[:, None]).min(0).values.expand(L, N).clone()
    hi = (f + 10 * sigma[:, None]).max(0).values.expand(L, N).clone()
    for _ in range(n_iter):
        mid = 0.5 * (lo + hi)
        F = STD_NORMAL.cdf((mid[None] - f[:, None, :]) / sigma[:, None, None]).mean(0)   # [L, N]
        below = F < probs[:, None]
        lo = torch.where(below, mid, lo)
        hi = torch.where(below, hi, mid)
    return 0.5 * (lo + hi)


def coverage_curve(u: Tensor, levels: Tensor) -> Tensor:
    """C(p) = (1/N) Σ_i 1{|u_i − 1/2| ≤ p/2}."""
    return ((u[None, :] - 0.5).abs() <= levels[:, None] / 2).to(u.dtype).mean(1)


REPORT_LEVELS = torch.tensor([0.50, 0.80, 0.90, 0.95], dtype=DTYPE)
CURVE_LEVELS  = torch.linspace(0.01, 0.99, 99, dtype=DTYPE)
N_test        = y_test.shape[0]

pit = {}
for stem, (f, sig) in PRED.items():
    u      = pit_values(f, sig, y_test)
    probs  = torch.cat([(1 - REPORT_LEVELS) / 2, (1 + REPORT_LEVELS) / 2])
    q      = predictive_quantiles(f, sig, probs)
    k      = REPORT_LEVELS.shape[0]
    width  = (q[k:] - q[:k]).mean(1) * y_std                   # W(p) · s_y
    pit[stem] = dict(
        u      = u,
        cov    = coverage_curve(u, REPORT_LEVELS),
        width  = width,
        curve  = coverage_curve(u, CURVE_LEVELS),
    )

# --- table -------------------------------------------------------------------
hdr = "".join(f"  C({p:.2f})  W({p:.2f})" for p in REPORT_LEVELS.tolist())
print(f"{'sampler':<24}{hdr}")
for stem, r in pit.items():
    row = "".join(f"  {c:7.3f}  {w:7.3f}" for c, w in zip(r["cov"].tolist(), r["width"].tolist()))
    print(f"{SAMPLER_LABELS.get(stem, stem):<24}{row}")
se = [math.sqrt(p * (1 - p) / N_test) for p in REPORT_LEVELS.tolist()]
print(f"{'± 1 se under calibration':<24}" + "".join(f"  {s:7.3f}         " for s in se))

# --- plots: calibration curve and PIT histogram -------------------------------
fig, (ax_c, ax_h) = plt.subplots(1, 2, figsize=(12, 4.5))
p_np  = CURVE_LEVELS.numpy()
band  = 2 * np.sqrt(p_np * (1 - p_np) / N_test)
ax_c.fill_between(p_np, -band, band, color="grey", alpha=0.2, label="±2 se (binomial)")
ax_c.axhline(0, color="k", lw=0.8)
bins = np.linspace(0, 1, 11)
for stem, r in pit.items():
    kw = dict(color=COLORS.get(stem, "grey"), ls=LINESTYLES.get(stem, "-"), label=SAMPLER_LABELS.get(stem, stem))
    ax_c.plot(p_np, r["curve"].numpy() - p_np, **kw)
    ax_h.hist(r["u"].numpy(), bins=bins, density=True, histtype="step", lw=1.5,
              color=kw["color"], ls=kw["ls"], label=kw["label"])
ax_c.set(xlabel="nominal level p", ylabel="C(p) − p", title="Calibration of central predictive intervals")
ax_h.axhline(1, color="k", lw=0.8)
ax_h.set(xlabel="u = F(y)", ylabel="density", title="PIT histogram (uniform = calibrated)")
ax_c.legend(fontsize=8)
plt.tight_layout()
plt.show()


## CONVERGENCE

In [ ]:
# ---------------------------------------------------------------------------
# INVARIANT / IDENTIFIABLE STATISTICS per posterior draw (convergence inputs)
# ---------------------------------------------------------------------------
# The sampled state is z = (β, log σ), with β ∈ R^{D−1} the network parameters.
# The likelihood is invariant under a group G of reparametrisations of β that
# leave f_β unchanged: permutation of hidden units within a layer, and, for tanh,
# sign flips  (w_in, b, w_out) → (−w_in, −b, −w_out)  since tanh(−a) = −tanh(a).
# The prior N(0, diag(λ)⁻¹) has a constant precision λ within each weight matrix
# and bias vector, so it is G-invariant too. The posterior is therefore
# G-invariant: every marginal of a single β_d is a symmetric mixture over
# ≥ |G| equivalent modes, and diagnostics on raw β_d measure mode-hopping over
# G, which is neither needed nor expected. Diagnostics are computed on
# statistics T(z) with T(g·z) = T(z) for all g ∈ G:
#
#   log_sigma      log σ_s
#   f_test         f_s(x_j),  j = 1..N_test                     (vector-valued)
#   loglik_train   ℓ_s = Σ_i log N(y_i ; f_s(x_i), σ_s²),        i over training set
#   log_post       log π(z_s) = ℓ_s − ½ Σ_d λ_d β_{s,d}² − ½ (σ_s / s_σ)² + log σ_s  + const,
#                  i.e. the unnormalised log target in the coordinates sampled
#                  (σ ~ HalfNormal(s_σ), with Jacobian of σ = e^{log σ})
#   rmse_train     sqrt( (1/n) Σ_i (y_i − f_s(x_i))² ) · s_y
#   lpd_test       (1/N_test) Σ_j log N(y_j ; f_s(x_j), σ_s²)   (per draw, not the mixture)
#   fro_<layer>    ‖W_{s,l}‖_F for each weight matrix W_l  (invariant: ‖P W Q‖_F = ‖W‖_F
#                  for permutation / signed-permutation matrices P, Q)
#   zero_frac      (1/(D−1)) Σ_d 1{β_{s,d} = 0}   (sticky samplers: fraction frozen at 0)
#
# Each scalar statistic is a series T(z_s), s = 1..S, in sampling order, to be
# reshaped to [chains, draws] for split-R̂ and ESS.
# ---------------------------------------------------------------------------
import time

X_train = data["X_train"]
y_train = data["y_train"]

_names   = [n for n, _ in bm.module.named_parameters()]
_shapes  = [p.shape for _, p in bm.module.named_parameters()]
_numels  = [int(s.numel()) for s in _shapes]
_offsets = np.cumsum([0] + _numels)
WEIGHT_SLICES = {n: slice(int(_offsets[k]), int(_offsets[k + 1]))
                 for k, n in enumerate(_names) if n.endswith("weight")}

LAMBDA   = bm.prior_precision[:-1]           # λ_d on β (the log σ entry is zero by construction)
S_SIGMA  = float(bm.prior_sigma_scale)
LOG_2PI  = math.log(2 * math.pi)


@torch.no_grad()
def forward_batched(beta: Tensor, X: Tensor, chunk: int = 500) -> Tensor:
    """f_s(X) for all draws, vectorised over s.   β: [S, D−1] -> [S, n]."""
    f1 = lambda b: torch.func.functional_call(bm.module, bm.param_dict_fn(b), (X,)).squeeze(-1)
    return torch.cat([torch.func.vmap(f1)(beta[i:i + chunk]) for i in range(0, beta.shape[0], chunk)])


def gauss_loglik(y: Tensor, f: Tensor, log_sigma: Tensor) -> Tensor:
    """Σ_i log N(y_i ; f_{s,i}, σ_s²)   -> [S]."""
    r2 = ((y[None, :] - f) ** 2).sum(1)
    n  = y.shape[0]
    return -0.5 * n * LOG_2PI - n * log_sigma - 0.5 * r2 / (2 * log_sigma).exp()


def invariant_stats(samples: Tensor, f_test: Tensor) -> tuple[dict[str, Tensor], dict[str, float]]:
    beta, log_sigma = samples[:, :-1], samples[:, -1]
    sigma = log_sigma.exp()
    timing, out = {}, {}

    t = time.perf_counter()
    f_train = forward_batched(beta, X_train)
    timing["forward_train"] = time.perf_counter() - t

    t = time.perf_counter()
    out["log_sigma"]    = log_sigma
    out["f_test"]       = f_test
    out["loglik_train"] = gauss_loglik(y_train, f_train, log_sigma)
    out["log_post"]     = (out["loglik_train"]
                           - 0.5 * (beta ** 2 * LAMBDA).sum(1)
                           - 0.5 * (sigma / S_SIGMA) ** 2 + log_sigma)
    out["rmse_train"]   = ((y_train[None, :] - f_train) ** 2).mean(1).sqrt() * y_std
    out["lpd_test"]     = gauss_loglik(y_test, f_test, log_sigma) / y_test.shape[0]
    for name, sl in WEIGHT_SLICES.items():
        out[f"fro_{name}"] = beta[:, sl].norm(dim=1)
    out["zero_frac"]    = (beta == 0).to(beta.dtype).mean(1)
    timing["statistics"] = time.perf_counter() - t
    return out, timing


INVAR_RUNS = {                 # stem -> (z_s [S, D], f_s(x_test) [S, N_test])
    "grid_zigzag":           (zz_samples,     zz_preds),
    "grid_boomerang":        (boom_samples,   boom_preds),
    "grid_sticky_zigzag":    (s_zz_samples,   s_zz_preds),
    "grid_sticky_boomerang": (s_boom_samples, s_boom_preds),
    "nuts":                  (nuts_samples,   nuts_preds),
    "lbbnn":                 (lbbnn_samples,  lbbnn_preds),
}

invar = {}
for stem, (z, f_te) in INVAR_RUNS.items():
    invar[stem], timing = invariant_stats(z, f_te)
    t_str = "  ".join(f"{k}={v:.2f}s" for k, v in timing.items())
    print(f"{SAMPLER_LABELS.get(stem, stem):<24} {t_str}")

# LBBNN noise: plug-in MLE of a shared σ given the variational draws of f, training data only.
#   σ̂² = (1/(S n)) Σ_s Σ_i (y_i − f_s(x_i))²
lbbnn_f_train = forward_batched(lbbnn_weight_samples, X_train)          # [S, n]
lbbnn_sigma_hat = ((y_train[None, :] - lbbnn_f_train) ** 2).mean().sqrt()
lbbnn_sigma = lbbnn_sigma_hat.expand(lbbnn_weight_samples.shape[0])    # [S]
print(f"LBBNN σ: fixed = 0.253,  plug-in σ̂ = {float(lbbnn_sigma_hat):.4f}")

# --- summary table: posterior mean (sd) of each scalar statistic --------------
scalar_keys = [k for k in next(iter(invar.values())) if k != "f_test"]
print()
print(f"{'statistic':<22}" + "".join(f"{SAMPLER_LABELS.get(s, s):>24}" for s in invar))
for k in scalar_keys:
    cells = "".join(f"{float(v[k].mean()):>13.4g} ({float(v[k].std()):.2g})".rjust(24) for v in invar.values())
    print(f"{k:<22}{cells}")


In [ ]:
# ---------------------------------------------------------------------------
# EFFECTIVE SAMPLE SIZE of the invariant statistics
# ---------------------------------------------------------------------------
# For a statistic T with draws T_{c,s}, c = 1..C chains, s = 1..S_c in sampling
# order, and autocorrelation ρ_t at lag t,
#
#     ESS  =  C·S / τ,        τ = 1 + 2 Σ_{t≥1} ρ_t    (integrated autocorrelation time),
#
# with ρ_t estimated jointly over chains and the sum truncated by Geyer's
# initial monotone sequence. Following Vehtari et al. (2021):
#   bulk-ESS   ESS of the rank-normalised draws  Φ⁻¹((r_{c,s} − 3/8) / (CS + 1/4)),
#              which measures efficiency for the centre of the distribution;
#   tail-ESS   min over q ∈ {0.05, 0.95} of the ESS of the indicator 1{T ≤ Q_q},
#              which measures efficiency for the 5% and 95% quantiles.
#
# Layout: PDMP runs are one continuous trajectory, resampled at sorted times
# after burn-in and thinned at equal index spacing, so C = 1 with draws in
# time order. NUTS is stored as C = 4 chains concatenated row-major,
# i.e. draw k belongs to chain ⌊k / S_c⌋.
#
# Cost-normalised efficiency:  ESS / G · 10³,  G = total gradient evaluations.
#
# For vector-valued f_test, ESS is computed per test point j and summarised by
# min_j and median_j.
# ---------------------------------------------------------------------------
import arviz as az

N_CHAINS = {"nuts": 4}          # stems not listed: C = 1


def to_chains(x: Tensor, C: int) -> np.ndarray:
    """[C·S_c, ...] -> [C, S_c, ...]."""
    x = x.detach().cpu().numpy()
    return x.reshape((C, -1) + x.shape[1:])


ess = {}
for stem, stats in invar.items():
    C  = N_CHAINS.get(stem, 1)
    ds = az.convert_to_dataset({k: to_chains(v, C) for k, v in stats.items()})
    bulk = az.ess(ds, method="bulk")
    tail = az.ess(ds, method="tail")
    G    = runs[stem]["run"].get("gradient_evals")
    ess[stem] = {}
    for k in stats:
        b, t = np.asarray(bulk[k].values), np.asarray(tail[k].values)
        ess[stem][k] = dict(
            bulk     = float(b.min()),
            tail     = float(t.min()),
            bulk_med = float(np.median(b)),
            tail_med = float(np.median(t)),
            bulk_per_kgrad = float(b.min()) / G * 1e3 if G else float("nan"),
        )

# --- table: bulk / tail ESS (f_test: min over test points, median in brackets) ---
stems = list(ess)
S_tot = {s: invar[s]["log_sigma"].shape[0] for s in stems}
print(f"{'statistic':<22}" + "".join(f"{SAMPLER_LABELS.get(s, s):>26}" for s in stems))
print(f"{'(draws)':<22}" + "".join(f"{S_tot[s]:>26d}" for s in stems))
for k in invar[stems[0]]:
    if k == "f_test":
        cells = "".join(
            f"{e[k]['bulk']:.0f}/{e[k]['tail']:.0f} ({e[k]['bulk_med']:.0f}/{e[k]['tail_med']:.0f})".rjust(26)
            for e in ess.values())
    else:
        cells = "".join(f"{e[k]['bulk']:.0f}/{e[k]['tail']:.0f}".rjust(26) for e in ess.values())
    print(f"{k:<22}{cells}")

# --- table: bulk-ESS per 1000 gradient evaluations ------------------------------
print()
print(f"{'bulk-ESS / 10³ grads':<22}" + "".join(f"{SAMPLER_LABELS.get(s, s):>26}" for s in stems))
for k in invar[stems[0]]:
    print(f"{k:<22}" + "".join(f"{ess[s][k]['bulk_per_kgrad']:>26.3g}" for s in stems))
